# L5: Reranking Tradeoffs, Failure Analysis, and Evaluation

这份 notebook 是对 `2-2-BuildingAndEvaluatingAdvancedRAGApplications` 中 reranking 部分的深入补全。

前面的内容已经让你知道：

- 先召回，再用 reranker 重排

但更深的问题还包括：

- `cross-encoder` 与 `bi-encoder` 的系统差异是什么
- latency / cost 的 tradeoff 怎么看
- rerank 会不会失败，如果会，失败长什么样
- online evaluation / A/B test 应该怎么设计


## 这一节的技术在做什么

`Reranking` 不是简单“再排一次序”，它实际上是在解决检索系统中的一个经典问题：

- 第一阶段召回往往快，但不够精细
- 第二阶段判断更细，但更贵、更慢

所以它天然就是一个系统工程问题，而不只是一个模型调用问题。

这一节会重点拆开讲四件事：

1. `bi-encoder` 为什么适合大规模召回
2. `cross-encoder` 为什么更适合精排
3. latency / quality / cost 之间的平衡
4. 怎么分析失败案例，怎么做在线 A/B 评估


## Imports


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Build a small corpus adapted from the existing advanced RAG lessons


In [ ]:
passages = [
    {
        "passage_id": "p1",
        "text": "Building a career in AI usually requires strong fundamentals in math, coding, and machine learning systems.",
        "label": "career_fundamentals",
    },
    {
        "passage_id": "p2",
        "text": "A common AI career path starts with software engineering, data work, or research before specializing in large models and deployment.",
        "label": "career_path",
    },
    {
        "passage_id": "p3",
        "text": "Evaluation is critical in advanced RAG because retrieval quality, groundedness, and answer usefulness all need separate measurement.",
        "label": "rag_evaluation",
    },
    {
        "passage_id": "p4",
        "text": "Rerankers refine a candidate set after retrieval by using a more precise but more expensive relevance model.",
        "label": "reranking",
    },
    {
        "passage_id": "p5",
        "text": "Sentence-window retrieval improves context quality by retrieving short spans and then expanding them into richer windows for the LLM.",
        "label": "sentence_window",
    },
    {
        "passage_id": "p6",
        "text": "Auto-merging retrieval combines child chunks into larger parents so the final context preserves more coherent structure.",
        "label": "auto_merging",
    },
]

passage_texts = [item["text"] for item in passages]
pd.DataFrame(passages)


## 2. Load a bi-encoder and a cross-encoder


In [ ]:
def find_snapshot(model_dir_name: str) -> Path:
    snapshot_root = Path("/Users/a1-6/Desktop/AIAgent/models") / model_dir_name / "snapshots"
    snapshots = sorted(
        path for path in snapshot_root.iterdir()
        if path.is_dir() and (path / "config.json").exists()
    )
    if not snapshots:
        raise FileNotFoundError(f"No valid snapshot found under {snapshot_root}")
    return snapshots[0]


bi_encoder = SentenceTransformer(str(find_snapshot("models--BAAI--bge-small-en-v1.5")))
cross_encoder = CrossEncoder(str(find_snapshot("models--BAAI--bge-reranker-base")))

passage_embeddings = bi_encoder.encode(passage_texts, normalize_embeddings=True)


## 3. Compare the retrieval stages


In [ ]:
def bi_encoder_retrieve(query: str, top_k: int = 4) -> pd.DataFrame:
    # bi-encoder 的优点是：可以把 query 和 passages 各自独立编码。
    # 这让它非常适合大规模向量召回，因为 passage embeddings 可以提前离线缓存。
    query_embedding = bi_encoder.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, passage_embeddings)[0]

    rows = []
    for passage, score in zip(passages, scores):
        rows.append({**passage, "bi_score": float(score)})

    return pd.DataFrame(rows).sort_values("bi_score", ascending=False).head(top_k).reset_index(drop=True)


def cross_encoder_rerank(query: str, candidates_df: pd.DataFrame) -> pd.DataFrame:
    # cross-encoder 的特点是：query 和 candidate 会一起送进模型重新打分。
    # 它不适合先给整个大库打分，但非常适合对“少量候选集”做高精度重排。
    pairs = [(query, row["text"]) for _, row in candidates_df.iterrows()]
    scores = cross_encoder.predict(pairs)

    reranked = candidates_df.copy()
    reranked["cross_score"] = scores
    return reranked.sort_values("cross_score", ascending=False).reset_index(drop=True)


In [ ]:
demo_query = "Why do advanced RAG systems use reranking after retrieval?"

bi_result = bi_encoder_retrieve(demo_query, top_k=4)
reranked_result = cross_encoder_rerank(demo_query, bi_result)

print("Bi-encoder top-k:")
print(bi_result[["passage_id", "label", "bi_score"]])
print("\nCross-encoder reranked top-k:")
print(reranked_result[["passage_id", "label", "bi_score", "cross_score"]])


## 4. Measure latency and discuss cost tradeoff


In [ ]:
benchmark_queries = [
    "What does reranking do in a RAG system?",
    "How does sentence-window retrieval improve context quality?",
    "What are common foundations for a career in AI?",
    "Why is evaluation important in advanced RAG?",
]

rows = []
for query in benchmark_queries:
    start = time.perf_counter()
    bi_candidates = bi_encoder_retrieve(query, top_k=4)
    bi_time = time.perf_counter() - start

    start = time.perf_counter()
    reranked = cross_encoder_rerank(query, bi_candidates)
    rerank_time = time.perf_counter() - start

    rows.append(
        {
            "query": query,
            "bi_retrieval_seconds": bi_time,
            "cross_rerank_seconds": rerank_time,
            "total_two_stage_seconds": bi_time + rerank_time,
            "top_result_after_rerank": reranked.iloc[0]["label"],
        }
    )

latency_df = pd.DataFrame(rows)
latency_df


## 5. Analyze failure cases


In [ ]:
# rerank 不是永远更好。下面专门造几个“容易混淆”的 query 来看失败模式。
failure_queries = [
    {
        "query": "What helps preserve structure when combining chunks?",
        "expected_label": "auto_merging",
    },
    {
        "query": "How do short retrieved spans become richer context windows?",
        "expected_label": "sentence_window",
    },
    {
        "query": "Why use a more expensive model after retrieval?",
        "expected_label": "reranking",
    },
]

failure_rows = []
for item in failure_queries:
    bi_candidates = bi_encoder_retrieve(item["query"], top_k=4)
    reranked = cross_encoder_rerank(item["query"], bi_candidates)
    top_before = bi_candidates.iloc[0]["label"]
    top_after = reranked.iloc[0]["label"]
    failure_rows.append(
        {
            "query": item["query"],
            "expected_label": item["expected_label"],
            "top_before_rerank": top_before,
            "top_after_rerank": top_after,
            "rerank_helped": top_before != item["expected_label"] and top_after == item["expected_label"],
            "rerank_hurt": top_before == item["expected_label"] and top_after != item["expected_label"],
        }
    )

pd.DataFrame(failure_rows)


## 6. Simulate an online A/B test design


In [ ]:
# 这里不是真的上线，而是模拟“如果要做线上 A/B test，该怎么记录指标”。
# A 组：只用 bi-encoder 召回后的 top1
# B 组：bi-encoder + cross-encoder rerank
ab_eval_queries = [
    {"query": "What is reranking used for?", "expected_label": "reranking"},
    {"query": "How do you build AI career fundamentals?", "expected_label": "career_fundamentals"},
    {"query": "What expands a short retrieval into richer context?", "expected_label": "sentence_window"},
    {"query": "What combines child chunks into larger parents?", "expected_label": "auto_merging"},
]

ab_rows = []
for item in ab_eval_queries:
    bi_candidates = bi_encoder_retrieve(item["query"], top_k=4)
    reranked = cross_encoder_rerank(item["query"], bi_candidates)

    a_prediction = bi_candidates.iloc[0]["label"]
    b_prediction = reranked.iloc[0]["label"]

    ab_rows.append(
        {
            "query": item["query"],
            "expected_label": item["expected_label"],
            "arm_a_hit": a_prediction == item["expected_label"],
            "arm_b_hit": b_prediction == item["expected_label"],
        }
    )

ab_df = pd.DataFrame(ab_rows)
print("Arm A accuracy:", ab_df["arm_a_hit"].mean())
print("Arm B accuracy:", ab_df["arm_b_hit"].mean())
ab_df
